# Amazon Electronics Intelligence

A Data Science project analyzing product reviews from the Amazon Electronics category.

**Dataset:** Amazon Electronics 5-core (reviews_Electronics_5.json)  
**Source:** [Julian McAuley, UCSD](http://jmcauley.ucsd.edu/data/amazon/)

In [ ]:
# Required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

---
## 1. Data Loading & Initial Overview

The dataset contains product reviews from the Amazon Electronics category. Each row is a JSON object (line-delimited JSON format).

In [ ]:
# Load the dataset
df = pd.read_json('../../reviews_Electronics_5.json/Electronics_5.json', lines=True)
print(f'Dataset loaded successfully!')
print(f'Number of rows: {df.shape[0]:,}')
print(f'Number of columns: {df.shape[1]}')

In [ ]:
# First 5 rows
df.head()

In [ ]:
# Column types and null info
df.info()

In [ ]:
# Summary statistics for numerical columns
df.describe()

### Column Descriptions

| Column | Description |
|--------|-------------|
| `reviewerID` | Unique identifier of the reviewer |
| `asin` | Amazon Standard Identification Number (unique product ID) |
| `reviewerName` | Display name of the reviewer |
| `helpful` | Helpfulness votes - [number of helpful votes, total votes] |
| `reviewText` | Full text of the review |
| `overall` | Product rating (1.0 - 5.0) |
| `summary` | Short title/summary of the review |
| `unixReviewTime` | Timestamp of the review (Unix epoch) |
| `reviewTime` | Date of the review (human-readable format) |

In [ ]:
# Missing value summary
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing Rate (%)': missing_pct
})

print('Missing Value Summary:')
print('=' * 40)
missing_df

In [ ]:
# Unique value counts
print('Unique Values per Column:')
print('=' * 40)
for col in df.columns:
    print(f'{col:20s} -> {df[col].nunique():>10,} unique values')

---
## 2. Descriptive Statistics

Getting the big picture of the dataset through rating distribution, most reviewed products, and most active reviewers.

### 2.1 Rating Distribution

In [ ]:
# Rating distribution
rating_counts = df['overall'].value_counts().sort_index()

fig, ax = plt.subplots(1, 2, figsize=(14,5))

# Bar chart
colors = ['#9467bd', '#8c564b', '#7f7f7f', '#1f77b4', '#d62728']
ax[0].bar(rating_counts.index, rating_counts.values, color=colors , edgecolor='black', linewidth=0.5)
ax[0].set_title('Rating Distribution', fontsize=14, fontweight='bold')
ax[0].set_xlabel('Rating', fontsize=12)
ax[0].set_ylabel('Number of Reviews', fontsize=12)
ax[0].set_xticks([1, 2, 3, 4, 5])

# Percentage pie chart
for i, v in enumerate(rating_counts.values):
    offset = rating_counts.max() * 0.01
    ax[0].text(rating_counts.index[i], v + offset, f'{v:,}', ha='center', fontsize=10)
    
ax[1].pie(rating_counts.values, labels=[f'{int(r)} Star' for r in rating_counts.index], autopct='%1.1f%%',
       colors=colors, startangle=90, textprops={'fontsize': 11})
ax[1].set_title('Rating Percentage', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


print(f'Average Rating: {df["overall"].mean():.2f}')
print(f'Median Rating: {df["overall"].median():.1f}')
print(f'Mode Rating: {df["overall"].mode()[0]:.1f}')

### 2.2 Most Reviewed Products (Top 10)

In [ ]:
# Top 10 most reviewed products
top_products = df['asin'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(14,5))

ax.barh(range(len(top_products)), top_products.values, color=sns.color_palette('viridis',10))
ax.set_yticks(range(len(top_products)))
ax.set_yticklabels(top_products.index, fontsize=10)
ax.set_xlabel('Number of Reviews', fontsize=12)
ax.set_title('Top 10 Most Reviewed Products' ,fontsize=14, fontweight='bold')
ax.invert_yaxis()

offset=top_products.max()*0.01
for i,v in enumerate(top_products.values):
    ax.text(v + offset, i, f'{v:,}', va='center', fontsize=10)
    
plt.tight_layout()
plt.show()

print(f'Total unique products: {df["asin"].nunique():,}')
print(f'Average reviews per product: {df.groupby("asin").size().mean():.1f}')


### 2.3 Most Active Reviewers (Top 10)

In [ ]:
# Top 10 most active reviewers
top_reviewers = df['reviewerID'].value_counts().head(10)

# Get reviewer names where available
reviewer_names = df.drop_duplicates('reviewerID').set_index('reviewerID')['reviewerName']
labels = [f'{reviewer_names.get(rid, "Unknown")} ({rid[:6]}...)' for rid in top_reviewers.index]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(top_reviewers)), top_reviewers.values, color=sns.color_palette('magma', 10))
ax.set_yticks(range(len(top_reviewers)))
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel('Number of Reviews', fontsize=12)
ax.set_title('Top 10 Most Active Reviewers', fontsize=14, fontweight='bold')
ax.invert_yaxis()

offset=top_reviewers.max()*0.01
for i, v in enumerate(top_reviewers.values):
    ax.text(v + offset, i, f'{v:,}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print(f'Total unique reviewers: {df["reviewerID"].nunique():,}')
print(f'Average reviews per user: {df.groupby("reviewerID").size().mean():.1f}')

---
## 3. Temporal Analysis

Electronics moves fast — let's see how time shapes the review landscape.

In [ ]:
# Convert unix timestamp to datetime and extract time components
df['review_date'] = pd.to_datetime(df['unixReviewTime'], unit='s')
df['review_year'] = df['review_date'].dt.year
df['review_month'] = df['review_date'].dt.month
df['review_ym'] = df['review_date'].dt.to_period('M')

print(f"Reviews span from {df['review_date'].min():%Y-%m-%d} to {df['review_date'].max():%Y-%m-%d}")
print(f"That's {(df['review_date'].max() - df['review_date'].min()).days:,} days of data")

### 3.1 Yearly Review Volume

In [ ]:
# Count reviews per year and calculate year-over-year growth
yearly = df.groupby('review_year').size().reset_index(name='count')
yearly['growth'] = yearly['count'].pct_change() * 100

fig, ax1 = plt.subplots(figsize=(14, 6))

# Bar chart: review volume
bars = ax1.bar(yearly['review_year'], yearly['count'],
               color=sns.color_palette('Blues_d', len(yearly)),
               edgecolor='black', lw=0.5)
ax1.set_ylabel('Reviews', fontsize=12, color='steelblue')
ax1.set_xlabel('Year')

# Add count labels
for b, c in zip(bars, yearly['count']):
    ax1.text(b.get_x() + b.get_width()/2, b.get_height(),
             f'{c:,}', ha='center', va='bottom', fontsize=9)

# Line chart: YoY growth rate
ax2 = ax1.twinx()
ax2.plot(yearly['review_year'].iloc[1:], yearly['growth'].iloc[1:],
         'o-', color='crimson', lw=2, label='YoY Growth %')
ax2.set_ylabel('Growth %', color='crimson')
ax2.axhline(0, color='gray', ls='--', alpha=0.4)
ax2.legend(loc='upper left')

ax1.set_title('Review Volume & Growth by Year', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

yearly

### 3.2 Seasonality

Do November (Black Friday) and December show any spikes in reviews or shifts in ratings?

In [ ]:
# Aggregate review volume and average rating by month
monthly_vol = df.groupby('review_month').size()
monthly_rating = df.groupby('review_month')['overall'].mean()
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Highlight Nov and Dec with different colors
vol_colors = ['#2196F3']*10 + ['#FF5722', '#E91E63']
rat_colors = ['#4CAF50']*10 + ['#FF5722', '#E91E63']

# Left: total reviews per month
ax1.bar(range(1,13), monthly_vol.values, color=vol_colors, edgecolor='black', lw=0.5)
ax1.set_xticks(range(1,13)); ax1.set_xticklabels(months)
ax1.set_title('Review Volume by Month', fontweight='bold')
ax1.set_ylabel('Total Reviews')
for i, v in enumerate(monthly_vol.values):
    ax1.text(i+1, v, f'{v:,}', ha='center', va='bottom', fontsize=8)

# Right: average rating per month
ax2.bar(range(1,13), monthly_rating.values, color=rat_colors, edgecolor='black', lw=0.5)
ax2.set_xticks(range(1,13)); ax2.set_xticklabels(months)
ax2.set_title('Average Rating by Month', fontweight='bold')
ax2.set_ylabel('Avg Rating')
ax2.set_ylim(3.5, 5.0)
for i, v in enumerate(monthly_rating.values):
    ax2.text(i+1, v + 0.02, f'{v:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

# Compare Nov/Dec against overall averages
avg_monthly = monthly_vol.mean()
avg_rating = df['overall'].mean()
for m, name in [(11,'November'), (12,'December')]:
    vol_diff = (monthly_vol[m] - avg_monthly) / avg_monthly * 100
    rat_diff = monthly_rating[m] - avg_rating
    print(f"{name}: {monthly_vol[m]:,} reviews ({vol_diff:+.1f}% vs avg), "
          f"rating {monthly_rating[m]:.2f} ({rat_diff:+.3f} vs {avg_rating:.2f})")

In [ ]:
# Monthly review volume trend over the full time period
ym = df.groupby('review_ym').size()

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(ym.index.astype(str), ym.values, color='steelblue', lw=1)
ax.fill_between(range(len(ym)), ym.values, alpha=0.15, color='steelblue')

# Show one tick per year for readability
ticks = list(range(0, len(ym), 12))
ax.set_xticks(ticks)
ax.set_xticklabels([ym.index.astype(str)[i] for i in ticks], rotation=45)
ax.set_title('Monthly Review Volume Over Time', fontweight='bold', fontsize=14)
ax.set_ylabel('Reviews')
plt.tight_layout()
plt.show()

### 3.3 Product Lifespan

How long does a product stay "alive" in terms of receiving reviews?

In [ ]:
# Calculate time between first and last review for each product
lifespan = df.groupby('asin')['review_date'].agg(['min','max','count'])
lifespan.columns = ['first', 'last', 'n_reviews']
lifespan['days'] = (lifespan['last'] - lifespan['first']).dt.days
lifespan['months'] = lifespan['days'] / 30.44

# Only products with 2+ reviews (single-review products have lifespan=0 by definition)
ls = lifespan[lifespan['n_reviews'] >= 2].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: lifespan distribution with mean/median markers
ax1.hist(ls['months'], bins=50, color='teal', edgecolor='black', lw=0.5, alpha=0.8)
ax1.axvline(ls['months'].median(), color='red', ls='--', lw=2,
            label=f"Median: {ls['months'].median():.1f}m")
ax1.axvline(ls['months'].mean(), color='orange', ls='--', lw=2,
            label=f"Mean: {ls['months'].mean():.1f}m")
ax1.set_xlabel('Lifespan (months)')
ax1.set_ylabel('Products')
ax1.set_title('Product Review Lifespan Distribution', fontweight='bold')
ax1.legend()

# Right: scatter plot — more reviews = longer lifespan?
sample = ls.sample(min(5000, len(ls)), random_state=42)
ax2.scatter(sample['months'], sample['n_reviews'], alpha=0.3, s=10, c='purple')
ax2.set_xlabel('Lifespan (months)')
ax2.set_ylabel('Reviews')
ax2.set_title('Lifespan vs Review Count', fontweight='bold')

plt.tight_layout()
plt.show()

# Summary stats
print(f"{len(ls):,} products with 2+ reviews")
print(f"Mean lifespan: {ls['months'].mean():.1f} months")
print(f"Median lifespan: {ls['months'].median():.1f} months")
print(f"\n< 1 month:  {(ls['months'] < 1).sum():,} ({(ls['months'] < 1).mean()*100:.1f}%)")
print(f"< 6 months: {(ls['months'] < 6).sum():,} ({(ls['months'] < 6).mean()*100:.1f}%)")
print(f"< 1 year:   {(ls['months'] < 12).sum():,} ({(ls['months'] < 12).mean()*100:.1f}%)")
print(f"> 3 years:  {(ls['months'] > 36).sum():,} ({(ls['months'] > 36).mean()*100:.1f}%)")

In [ ]:
# Percentile breakdown of product lifespan
for p in [10, 25, 50, 75, 90, 95]:
    print(f"  p{p}: {np.percentile(ls['months'], p):.1f} months")

# Group products into lifespan categories
ls['bucket'] = pd.cut(ls['months'],
                      bins=[0,1,3,6,12,24,36,60,float('inf')],
                      labels=['<1m','1-3m','3-6m','6-12m','1-2y','2-3y','3-5y','5y+'],
                      right=False)

bc = ls['bucket'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(bc.index.astype(str), bc.values,
       color=sns.color_palette('YlOrRd', len(bc)), edgecolor='black', lw=0.5)
ax.set_title('Products by Review Lifespan', fontweight='bold', fontsize=14)
ax.set_ylabel('Products')

for i, v in enumerate(bc.values):
    ax.text(i, v, f'{v:,}\n({v/len(ls)*100:.1f}%)', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()